# OLMo-2-1B × Tulu-3 SFT mixture leaderboard — robustness pilot (9000 steps, ~225M slot-tokens)

Robustness pilot from `~/.claude/plans/as-part-of-our-tender-quilt.md`. Tests whether `adam-polar-product-lora-coupled-spectral-chord-tight` (plain k=1) keeps its eval-loss advantage over AdamW-LoRA when the dataset is swapped from opc-sft-stage2 (code-IFT) to Tulu-3 (general-IFT, chat-templated via the open-instruct Tulu chat template).

Cell: OLMo-2-1B × Tulu-3 SFT mixture (400k subset, ~150k packed slots @ seq=2048) × global_batch=16 (batch=4 × accum=4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4}
- **chord-tight k=1** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2}

Source log groups: `{adamw,chord_tight}_robustness_tulu3_1b_r{64,256}_blackwell` (4 groups total).

**σ anchor**: no per-dataset multi-seed AdamW run yet. Quoting Δ against `σ_AdamW(packed_v1, opc-sft-stage2, r=64) = 0.0017` as a **proxy only** — re-anchor before any paper claim.

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt

from lora_playground.plotting import leaderboard_panel, canonical_label

# Run membership comes from the shared registry (lora_playground.workloads) — the
# SAME source the leaderboard doc uses, so notebook and doc cannot drift. Cells are
# leaderboard_panel(model, dataset, rank, ...) + an optional label_filter(label, cfg).

def ns_of(cfg):
    oc = cfg.get('optimizer_config') or {}
    return cfg.get('muon_ns_steps', oc.get('ns_steps'))

def picard_of(cfg):
    return cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')) or 1

def is_curv(label):
    # curvature-whitening / SOAP-curv / KL-Shampoo arms.
    return ('SOAP-curv' in label) or ('KL-Shampoo' in label) or ('+curv' in label)

## r=64

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('allenai/OLMo-2-0425-1B', 'tulu3', 64,
    'Tulu-3 r=64',
    figsize=(11, 4))
plt.show()
sdf

## r=256

In [ ]:
_fig, _tdf, sdf = leaderboard_panel('allenai/OLMo-2-0425-1B', 'tulu3', 256,
    'Tulu-3 r=256',
    figsize=(11, 4))
plt.show()
sdf